In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# **Step 1 — Check Kaggle Environment**

In [2]:
# Check Python version

import sys

print("Python version:", sys.version)

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


# **Step 2 — Install the libraries**

In [3]:
!pip install -q transformers accelerate

# **Step 3 — Check Transformers**

In [4]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.0.0


# **Step 4 — Check GPU Availability**

In [5]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cpu
GPU available: False


# **Step 5 — Load a Conversational Model**

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,  # float32 becuase CPU is using
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully on:", device)

Loading model: Qwen/Qwen2.5-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cpu


# **Step 6 — Build the Chatbot Function**

In [7]:
# System prompt — you can change and check chatbot "personality" or can set business use-case
SYSTEM_PROMPT = "You are a helpful, friendly AI assistant for a business. Answer clearly and concisely."

# Conversation history here
conversation_history = [
    {"role": "system", "content": SYSTEM_PROMPT}
]

def chat(user_message, max_new_tokens=200):
    """Ek user message le kar, model se reply generate karta hai aur history update karta hai."""
    conversation_history.append({"role": "user", "content": user_message})

    # Chat template apply and make prompt
    prompt = tokenizer.apply_chat_template(
        conversation_history,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    # only new generated part
    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    conversation_history.append({"role": "assistant", "content": reply})
    return reply

# **Step 7 — Test the Chatbot**

In [8]:
response = chat("Hello! Who are you and what can you help me with?")
print("Bot:", response)

Bot: I am an AI language model designed to assist with various tasks such as generating text, answering questions, providing information, and more. I'm here to offer helpful responses and support on any topic you'd like. How can I assist you today?


In [9]:
response = chat("Can you give me 3 tips for improving customer service in a small business?")
print("Bot:", response)

Bot: Certainly! Here are three tips that can significantly enhance your customer service in a small business:

1. **Be Approachable and Friendly**: Always be ready to answer customers' questions or concerns promptly. Being approachable shows genuine interest in their needs and helps build trust.

2. **Provide Value**: Offer something unique or extra value beyond the basic necessities. This could include free samples, discounts on products, or special offers that justify why these items might not be available elsewhere.

3. **Follow Up Actively**: After handling a customer's inquiry or issue, make sure to follow up promptly. A quick response can often lead to positive word-of-mouth marketing and better retention of current customers.

These strategies will not only improve your customer experience but also contribute to maintaining customer loyalty and building a strong brand.


# **Step 8 — Interactive Chat Loop**

In [ ]:
print("Chatbot ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Bot: Goodbye!")
        break
    reply = chat(user_input)
    print("Bot:", reply)

Chatbot ready! Type 'exit' to stop.



You:  hello i am umar


Bot: Hello Umar! It’s nice to meet you. Is there anything specific you would like to discuss or ask about?


You:  can i earn money with vibe coding from freelancing?


Bot: Yes, it is possible to earn money through freelance coding projects. Freelance opportunities involve working remotely to provide services to clients who pay for work done. Here are some steps you might consider to get started:

1. **Identify Your Skills**: Determine which programming languages, technologies, or areas of expertise you are proficient in. Many platforms, including GitHub, have repositories where developers share their skills and work.

2. **Research Freelancer Platforms**: Look into different freelancer platforms such as Fiverr, Upwork, and Freelancer. Each platform has its own set of requirements, so tailor your profile accordingly.

3. **Create High-Quality Work**: Ensure that your code is well-organized, easy to understand, and optimized for performance. Good documentation and clear explanations can greatly benefit your potential clients.

4. **Offer Competitive Rates**: Be prepared to negotiate rates based on the quality of your work and the level of effort requi

You:  i have completed python course from early 2025


Bot: That's great to hear! Completing a Python course from early 2025 suggests you've taken significant steps toward becoming a professional developer. Here are a few next steps that could further enhance your skills and career prospects:

### 1. **Build a Portfolio**
- Create a portfolio showcasing your projects. Include links to your GitHub repositories, past projects, and any other relevant documents.
- Use platforms like GitHub or GitLab to host your files. This makes it easier to update them over time.

### 2. **Join Relevant Communities**
- Join online communities related to Python development. Sites like Stack Overflow, Reddit (r/learnpython), and others can be valuable resources for learning new things and networking.
- Participate in discussions and forums dedicated to software engineering and web development.

### 3. **Network and Learn from Others**
- Connect with experienced developers through online groups and forums. Attending workshops, webinars, and conferences can prov

# **Next Steps — Making This Business-Ready**

# **Step 9 — Load a Bigger Model (Optional)**

In [ ]:
# If you want better answers, you can switch to a bigger model.
# This works best if you have a GPU session enabled in Kaggle.
def load_model(model_name):
    # This function replaces the current model with a new one
    global tokenizer, model, device, MODEL_NAME

    MODEL_NAME = model_name
    print("Loading model:", MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print("Model loaded on:", device)

# Example: try a bigger model (needs GPU)
# load_model("Qwen/Qwen2.5-1.5B-Instruct")

# **Step 10 — Simple Knowledge Base Search (Basic RAG)**

In [ ]:
# A small list of business facts (add your own here)
KNOWLEDGE_BASE = [
    "Our office hours are Monday to Saturday, 9 AM to 6 PM.",
    "Products can be returned within 7 days with a receipt.",
    "Delivery usually takes 3 to 5 working days.",
    "Customer support is available on WhatsApp and email.",
]

def find_relevant_facts(user_message):
    # Very simple search: check if any word from the fact
    # also appears in the user's message
    message_words = set(user_message.lower().split())
    matches = []

    for fact in KNOWLEDGE_BASE:
        fact_words = set(fact.lower().split())
        if message_words & fact_words:  # if there is any common word
            matches.append(fact)

    return matches

def chat_with_context(user_message, max_new_tokens=200):
    # Find any relevant facts first
    relevant_facts = find_relevant_facts(user_message)

    if relevant_facts:
        facts_text = "\n".join(f"- {f}" for f in relevant_facts)
        full_message = f"Company info:\n{facts_text}\n\nQuestion: {user_message}"
    else:
        full_message = user_message

    return chat(full_message, max_new_tokens=max_new_tokens)

In [ ]:
response = chat_with_context("What are your office hours?")
print("Bot:", response)

# **Step 11 — Simple Web Interface with Gradio**

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def gradio_chat_fn(message, history):
    # Gradio passes the message here, we just reuse our existing chat function
    return chat_with_context(message)

demo = gr.ChatInterface(
    fn=gradio_chat_fn,
    title="Business AI Assistant",
    description="Ask me anything about the business!",
)

demo.launch(share=True)

# **Step 12 — Basic Guardrails**

In [ ]:
# Add words you want to block here
BLOCKED_WORDS = ["hack", "bomb", "weapon", "illegal"]

FALLBACK_MESSAGE = "Sorry, I can only help with business-related questions. How else can I assist you?"

def is_blocked(message):
    # Check if any blocked word appears in the message
    lowered = message.lower()
    for word in BLOCKED_WORDS:
        if word in lowered:
            return True
    return False

def safe_chat(user_message, max_new_tokens=200):
    # Check the message before sending it to the model
    if is_blocked(user_message):
        return FALLBACK_MESSAGE
    return chat_with_context(user_message, max_new_tokens=max_new_tokens)

In [ ]:
print(safe_chat("Can you give me tips for improving customer service?"))
print(safe_chat("How do I hack into a system?"))

# **Step 13 — Persistent Memory with SQLite**

In [ ]:
import sqlite3
import json

DB_PATH = "chat_history.db"

def init_db():
    # Creates the table if it doesn't already exist
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS conversations (
            session_id TEXT PRIMARY KEY,
            history TEXT
        )
    """)
    conn.commit()
    conn.close()

init_db()

In [ ]:
def save_history(session_id, history):
    # Converts the history list into text (JSON) and saves it under session_id
    conn = sqlite3.connect(DB_PATH)
    conn.execute(
        "INSERT OR REPLACE INTO conversations (session_id, history) VALUES (?, ?)",
        (session_id, json.dumps(history))
    )
    conn.commit()
    conn.close()

In [ ]:
def load_history(session_id):
    # Loads history for a session_id, or starts a fresh one if none exists
    conn = sqlite3.connect(DB_PATH)
    row = conn.execute(
        "SELECT history FROM conversations WHERE session_id = ?",
        (session_id,)
    ).fetchone()
    conn.close()

    if row:
        return json.loads(row[0])
    else:
        return [{"role": "system", "content": SYSTEM_PROMPT}]

In [ ]:
# Load (or start) a conversation for this user
conversation_history = load_history("user_123")

# Chat normally — this updates conversation_history in memory
response = chat("Hi, what services do you offer?")
print("Bot:", response)

# Save it to the database
save_history("user_123", conversation_history)

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))

In [ ]:
print(load_history("user_123"))